**Organizar Información Adicional deminerales**
En este cuaderno estructuro la información de los siguientes datos para manejarlos en STATA:
- Minería ilegal calculada por SR2021
- Producción minera asociada a regalías

# Setup

In [1]:
# define root path of project 
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

# load general setup
from utils.setup_general import *

# load GIS setup
from utils.setup_gis_python import *

# Load utils for SR 2021
from utils.utils_for_SR2021 import *

Setup general cargado
Setup GIS cargado


# Definir datos de salida
El archivo de salida se estructura para conservar la misma estructura entre todas las bases de datos

In [2]:
ORDEN_DF

['codigo_dane_municipio',
 'anno',
 'nombre_variable',
 'variable_sujeto',
 'variable_medicion',
 'variable_detalle',
 'variable_descripcion',
 'valor',
 'clasificacion_econometria']

In [3]:
# Definir DF con la estructura acordada para el proyecto
# La variable global ORDEN_DF ya tiene la lista de todas las columnas ordenadas
# Inicializar el DF vacío
df_salida = pd.DataFrame(
    columns=[ORDEN_DF]
)

# Datos de minería ilegal de SR2021

In [4]:
# Cargar datos
df_SR2021_minerales = pd.read_parquet(
    DATA/'intermediate/e2011_SR2021_mineriaIlegal_armonizado.parquet'   
)

# Datos de producción minera legal asociada a regalías

In [18]:
# Cargar datos
df_UPME_produccionRegalias = pd.read_parquet(
    DATA/'intermediate/e2011_UPME_produccionRegalias_armonizado.parquet'
)

print(MSC_SEPARADOR, "Datos originales")
# Exploración inicial
df_UPME_produccionRegalias.head()


-------------------------------- Datos originales


,Año,Mes,Mineral,UnidadMedida,Regalias ($),Produccion,Codigo Municipio,Municipio,Codigo Departamento,Departamento,texto_busqueda,categoria_armonizada
0,2026,1,ORO,GRAMOS,"1,013,843,150.81","634,108.02",5736,SEGOVIA,5,ANTIOQUIA,oro,Oro
1,2025,12,ORO,GRAMOS,"969,828,842.90","690,459.19",5736,SEGOVIA,5,ANTIOQUIA,oro,Oro
2,2025,11,ORO,GRAMOS,"746,090,262.90","612,155.49",5736,SEGOVIA,5,ANTIOQUIA,oro,Oro
3,2025,10,ORO,GRAMOS,"705,220,681.15","567,809.15",5736,SEGOVIA,5,ANTIOQUIA,oro,Oro
4,2025,9,ORO,GRAMOS,"885,010,981.44","788,920.45",5736,SEGOVIA,5,ANTIOQUIA,oro,Oro


## Procesamiento

In [36]:
# Conservar solo las columnas de interés
columnas_para_conservar = ['categoria_armonizada', 'Año', 'Codigo Municipio', 'Mineral', 'UnidadMedida', 'Regalias ($)', 'Produccion']
df_UPME_produccionRegalias = df_UPME_produccionRegalias[columnas_para_conservar]


# Asegurarse que la columna del codigo de municipio sea un string de 5 dígitos
df_UPME_produccionRegalias["Codigo Municipio"] = (
    df_UPME_produccionRegalias["Codigo Municipio"]
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)


# Agregar valores mensuales en valores anuales
df_UPME_produccionRegalias_anual = df_UPME_produccionRegalias.groupby(
    ['categoria_armonizada', 'Año', 'Codigo Municipio', 'Mineral', 'UnidadMedida'])[['Regalias ($)', 'Produccion']].sum().reset_index()


print(MSC_SEPARADOR, "Datos procesados")
display(df_UPME_produccionRegalias_anual.head())


-------------------------------- Datos procesados


,categoria_armonizada,Año,Codigo Municipio,Mineral,UnidadMedida,Regalias ($),Produccion
0,Carbon,2012,05030,CARBON,TONELADAS,"1,051,738,901.34","237,922.24"
1,Carbon,2012,05036,CARBON,TONELADAS,"54,852,809.23","6,826.35"
2,Carbon,2012,05282,CARBON,TONELADAS,"294,127,636.19","73,610.71"
3,Carbon,2012,05809,CARBON,TONELADAS,"695,137,060.68","181,140.02"
4,Carbon,2012,05861,CARBON,TONELADAS,"161,453,262.02","6,353.69"


In [38]:
# Mostrar los valores en los que se mide la producción de diferentes materiales
# Estoy explorando sie puedo sumar la producción de todos los mienerales dentro de la misma categoría armonizada
# Hay algunos en los que es posible y otros y en los que se podrían convertir unidades de volumen a unidades de masa
# por ahora, para la producción solo voy a agregar los que tengan la misma unidadd de medida
# En refinamientos posteriores podría hacer la conversión de unidades de la que hablo arriba.
with pd.option_context("display.max_rows", None):
    display(
        df_UPME_produccionRegalias_anual
        .groupby(["categoria_armonizada", "UnidadMedida"])["Mineral"]
        .value_counts()
    )

categoria_armonizada     UnidadMedida    Mineral                                                        
Carbon                   TONELADAS       CARBON                                                             1093
                                         CARBON TERMICO                                                      623
                                         CARBON METALURGICO                                                  479
                                         CARBON ANTRACITA                                                     13
Cobre                    KILOGRAMOS      COBRE                                                                58
Fertilizantes            TONELADAS       ROCA FOSFORICA                                                      119
Gemas                    QUILATES        ESMERALDAS EN BRUTO                                                  94
                                         ESMERALDAS TALLADAS                                            

## Valor de regalías

In [41]:
# Agregar valor de las regalías
df_UPME_produccionRegalias_valorRegalias = df_UPME_produccionRegalias_anual.groupby(
    ["categoria_armonizada", 'Año', 'Codigo Municipio'])['Regalias ($)'].sum().reset_index()

### Dar estructura al DF

In [68]:
# hacer copia de los datos
panel_valorRegalias = df_UPME_produccionRegalias_valorRegalias.copy()

# Asignar nombres acordados a las columnas
panel_valorRegalias = panel_valorRegalias.rename(
    columns={
        'categoria_armonizada':COL_VARIABLE_SUJETO,
        'Año':COL_ANNO,
        'Codigo Municipio': COL_ID_MUNICIPIO,
        'Regalias ($)': COL_VALOR
    }
)

# Completar información de estructura del panel
panel_valorRegalias[COL_CLASIFICACION_ECONOMETRIA] = 'X'
panel_valorRegalias[COL_NOMBRE_DE_VARIABLE] = ''
panel_valorRegalias[COL_VARIABLE_DETALLE] = 'Valor asociado a regalías en COP Corrientes'
panel_valorRegalias[COL_VARIABLE_MEDICION] = 'produccionRegalias_copCorrientes' # ¿Habría que pasarlo a plata constante?
panel_valorRegalias[COL_VARIABLE_DESCRIPCION] = ('Producción legal.' +
                                                ' Mineral: ' + panel_valorRegalias[COL_VARIABLE_SUJETO] +
                                                 ' Medicion: ' + panel_valorRegalias[COL_VARIABLE_MEDICION] +
                                                 ' Detalle: '+ panel_valorRegalias[COL_VARIABLE_DETALLE]
                                                )

display(panel_valorRegalias[ORDEN_DF])

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05030,2012,,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"1,051,738,901.34",X
1,05036,2012,,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"54,852,809.23",X
2,05282,2012,,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"294,127,636.19",X
3,05809,2012,,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"695,137,060.68",X
4,05861,2012,,Carbon,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"161,453,262.02",X
...,...,...,...,...,...,...,...,...,...
13575,73067,2026,,Otros metales preciosos,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Otros metales preci...,"167,580.49",X
13576,73168,2026,,Otros metales preciosos,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Otros metales preci...,"4,499.90",X
13577,73270,2026,,Otros metales preciosos,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Otros metales preci...,"318,733.95",X
13578,73411,2026,,Otros metales preciosos,produccionRegalias_copCorrientes,Valor asociado a regalías en COP Corrientes,Producción legal. Mineral: Otros metales preci...,"35,647,868.89",X


## Producción

In [53]:
# Agregar producción
# En cada "categoria_armonizada", voy a elegir el mayor conteo de registros de producción.
# Elijo esa unidad de medida y agrego en esa unidad de medida

# Econtrar el conteo de observaciones
unidades_de_medida_conteo = df_UPME_produccionRegalias_anual.groupby(["categoria_armonizada"])["UnidadMedida"].value_counts().reset_index()
display(unidades_de_medida_conteo)

# Conservar solo la unidad de medida con más registros en cada categoria armonziada
unidades_principales = (
    unidades_de_medida_conteo.loc[
        unidades_de_medida_conteo
        .groupby("categoria_armonizada")["count"]
        .idxmax()
    ]
    .reset_index(drop=True)
)
print(MSC_SEPARADOR, "Unidades principales")
display(unidades_principales)

,categoria_armonizada,UnidadMedida,count
0,Carbon,TONELADAS,2208
1,Cobre,KILOGRAMOS,58
2,Fertilizantes,TONELADAS,119
3,Gemas,QUILATES,254
4,Materiales construccion,METROS CUBICOS,11391
5,Materiales construccion,TONELADAS,1009
6,Materiales construccion,KILOGRAMOS,9
7,Metales base,TONELADAS,118
8,Metales base,KILOGRAMOS,71
9,Minerales industriales,TONELADAS,3013



-------------------------------- Unidades principales


,categoria_armonizada,UnidadMedida,count
0,Carbon,TONELADAS,2208
1,Cobre,KILOGRAMOS,58
2,Fertilizantes,TONELADAS,119
3,Gemas,QUILATES,254
4,Materiales construccion,METROS CUBICOS,11391
5,Metales base,TONELADAS,118
6,Minerales industriales,TONELADAS,3013
7,Niquel,LIBRAS,42
8,Oro,GRAMOS,2095
9,Otros,METROS CUBICOS,51


In [57]:
# Para la producción voy a agregar únicamente las categorías armonizadas con la misma uniadd de medida
# Desecho las demás observaciones

df_UPME_produccionRegalias_anual_mismaUnidad = (
    df_UPME_produccionRegalias_anual
    .merge(
        unidades_principales[
            ["categoria_armonizada", "UnidadMedida"]
        ],
        on=["categoria_armonizada", "UnidadMedida"],
        how="inner",
        validate="many_to_one"
    )
)

In [61]:
# Agregar Producción asociada a regalías

df_UPME_produccionRegalias_produccionAsociadaRegalias = df_UPME_produccionRegalias_anual.groupby(
    ["categoria_armonizada", 'Año', 'Codigo Municipio'])[['Produccion', 'UnidadMedida']].aggregate({'Produccion':'sum', 'UnidadMedida':'first'}).reset_index()
display(df_UPME_produccionRegalias_produccionAsociadaRegalias)

,categoria_armonizada,Año,Codigo Municipio,Produccion,UnidadMedida
0,Carbon,2012,05030,"237,922.24",TONELADAS
1,Carbon,2012,05036,"6,826.35",TONELADAS
2,Carbon,2012,05282,"73,610.71",TONELADAS
3,Carbon,2012,05809,"181,140.02",TONELADAS
4,Carbon,2012,05861,"6,353.69",TONELADAS
...,...,...,...,...,...
13575,Otros metales preciosos,2026,73067,681.85,GRAMOS
13576,Otros metales preciosos,2026,73168,0.00,GRAMOS
13577,Otros metales preciosos,2026,73270,0.00,GRAMOS
13578,Otros metales preciosos,2026,73411,"173,689.24",GRAMOS


### Dar estructura al DF

In [69]:
# hacer copia de los datos
panel_produccionAsociadaRegalias = df_UPME_produccionRegalias_produccionAsociadaRegalias.copy()

# Asignar nombres acordados a las columnas
panel_produccionAsociadaRegalias = panel_produccionAsociadaRegalias.rename(
    columns={
        'categoria_armonizada':COL_VARIABLE_SUJETO,
        'Año':COL_ANNO,
        'Codigo Municipio': COL_ID_MUNICIPIO,
        'Produccion': COL_VALOR,
        'UnidadMedida': COL_VARIABLE_MEDICION
    }
)

# Completar información de estructura del panel
panel_produccionAsociadaRegalias[COL_CLASIFICACION_ECONOMETRIA] = 'X'
panel_produccionAsociadaRegalias[COL_NOMBRE_DE_VARIABLE] = ''
panel_produccionAsociadaRegalias[COL_VARIABLE_DETALLE] = 'Producción asociada a regalías en COP Corrientes'
panel_produccionAsociadaRegalias[COL_VARIABLE_MEDICION] = 'produccionRegalias_' + panel_produccionAsociadaRegalias[COL_VARIABLE_MEDICION]
panel_produccionAsociadaRegalias[COL_VARIABLE_DESCRIPCION] = ('Producción legal.' +
                                                ' Mineral: ' + panel_produccionAsociadaRegalias[COL_VARIABLE_SUJETO] +
                                                 ' Medicion: ' + panel_produccionAsociadaRegalias[COL_VARIABLE_MEDICION] +
                                                 ' Detalle: '+ panel_produccionAsociadaRegalias[COL_VARIABLE_DETALLE]
                                                )

display(panel_produccionAsociadaRegalias[ORDEN_DF])

,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,05030,2012,,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"237,922.24",X
1,05036,2012,,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"6,826.35",X
2,05282,2012,,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"73,610.71",X
3,05809,2012,,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"181,140.02",X
4,05861,2012,,Carbon,produccionRegalias_TONELADAS,Producción asociada a regalías en COP Corrientes,Producción legal. Mineral: Carbon Medicion: pr...,"6,353.69",X
...,...,...,...,...,...,...,...,...,...
13575,73067,2026,,Otros metales preciosos,produccionRegalias_GRAMOS,Producción asociada a regalías en COP Corrientes,Producción legal. Mineral: Otros metales preci...,681.85,X
13576,73168,2026,,Otros metales preciosos,produccionRegalias_GRAMOS,Producción asociada a regalías en COP Corrientes,Producción legal. Mineral: Otros metales preci...,0.00,X
13577,73270,2026,,Otros metales preciosos,produccionRegalias_GRAMOS,Producción asociada a regalías en COP Corrientes,Producción legal. Mineral: Otros metales preci...,0.00,X
13578,73411,2026,,Otros metales preciosos,produccionRegalias_GRAMOS,Producción asociada a regalías en COP Corrientes,Producción legal. Mineral: Otros metales preci...,"173,689.24",X


# organizar panel de salida

In [ ]:
# Bases a exportar
panel_valorRegalias
panel_produccionAsociadaRegalias